# Key custody and refusals

Use real non-extractable WebCrypto keys and a DER-returning local custodian, with explicit encoding and custody refusals.

Install Deno and register its TypeScript kernel with `deno jupyter --install`. Open this file
in a Jupyter front end such as JupyterLab or VS Code, select **Deno**, then Run All in a fresh kernel.
The cells import exact published signer **2.0.0** and verifier **2.0.0**.

Fixed times describe demonstration credentials. Generated keys, signatures and nonces stay in
memory. Signing and verification do not grant operational authority; the host owns trust, live
revocation, replay and policy.


## Load the published packages

Check both release identities and the backend Deno selects.


In [ ]:
import assert from "node:assert/strict";
import { Buffer } from "node:buffer";
import { generateKeyPairSync, sign } from "node:crypto";
import * as signer from "npm:@bounded-authority-protocol/signer@2.0.0";
import * as verifier from "npm:@bounded-authority-protocol/verifier@2.0.0";
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.0/package.json" with { type: "json" };
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with { type: "json" };
assert.equal(signerMetadata.version, "2.0.0");
assert.equal(verifierMetadata.version, "2.0.0");
assert.equal(verifier.BACKEND_ID, "portable");
console.log(JSON.stringify({ signer: signerMetadata.version, verifier: verifierMetadata.version, backend: verifier.BACKEND_ID }));


{"signer":"2.0.0","verifier":"2.0.0","backend":"portable"}


## Prepare real demonstration key handles

Each constructor holds a fresh private key in its closure and exposes only the key-handle contract.
No key material is displayed. Production custody supplies this same contract.


In [ ]:
function createEd25519Handle(keyId: string, role: signer.KeyRole = "holder"): signer.KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ed25519");
  const publicBytes = new Uint8Array(Buffer.from(publicKey.export({ format: "jwk" }).x!, "base64url"));
  return {
    sign: (signingInput) => new Uint8Array(sign(null, signingInput, privateKey)),
    publicKey: () => publicBytes,
    thumbprint: () => verifier.thumbprint(verifier.jwkFromPublicKey(publicBytes)),
    keyIdentity: () => ({ keyId, publicKey: publicBytes }),
    signingIdentity: () => ({ role, keyId, publicKey: publicBytes }),
  };
}
function createEs256Handle(keyId: string, role: signer.KeyRole = "holder"): signer.Es256KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ec", { namedCurve: "P-256" });
  const { x, y } = publicKey.export({ format: "jwk" }) as { x: string; y: string };
  const publicBytes = new Uint8Array([0x04, ...Buffer.from(x, "base64url"), ...Buffer.from(y, "base64url")]);
  return {
    sign: (signingInput) => new Uint8Array(sign("sha256", signingInput, { key: privateKey, dsaEncoding: "ieee-p1363" })),
    publicKey: () => publicBytes,
    thumbprint: () => verifier.v3.thumbprint({ crv: "P-256", kty: "EC", x, y }),
    keyIdentity: () => ({ keyId, publicKey: publicBytes }),
    signingIdentity: () => ({ role, keyId, publicKey: publicBytes }),
  };
}
const ed25519Width = (await createEd25519Handle("example-ed25519").publicKey()).length;
const es256Width = (await createEs256Handle("example-es256").publicKey()).length;
assert.equal(ed25519Width, 32);
assert.equal(es256Width, 65);
console.log(JSON.stringify({ ed25519Bytes: ed25519Width, es256Bytes: es256Width }));


{"ed25519Bytes":32,"es256Bytes":65}


## Create a non-exporting Ed25519 issuer

The caller creates the real WebCrypto pair. Its private key is non-extractable.


In [ ]:
const issuerPair = await crypto.subtle.generateKey({ name: "Ed25519" }, false, ["sign", "verify"]) as CryptoKeyPair;
const issuerHandle = await signer.ed25519KeyHandle(issuerPair, { keyId: "webcrypto-issuer", role: "issuer" });
assert.ok(issuerHandle.ok);
assert.equal(issuerPair.privateKey.extractable, false);
console.log(JSON.stringify({ ok: issuerHandle.ok, extractable: issuerPair.privateKey.extractable }));


{"ok":true,"extractable":false}


## Sign with the WebCrypto issuer

The signer receives the handle while private key custody remains with WebCrypto.


In [ ]:
const holder = createEd25519Handle("calendar-agent");
const calendarGrant = await signer.signGrant({ issuer: "https://calendar.example", grantId: "calendar-grant",
  audiences: ["https://calendar.example"], issuedAt: 1_000, notBefore: 1_000, expiresAt: 2_000,
  holderThumbprint: await holder.thumbprint(), operations: [{ name: "create_event", selectors: [{ kind: "all" }] }],
}, issuerHandle.value);
assert.ok(calendarGrant.ok);
console.log(JSON.stringify({ ok: calendarGrant.ok }));


{"ok":true}


## Create a non-exporting P-256 handle

Major 3 uses ECDSA P-256 and the same caller-owned key lifetime.


In [ ]:
const ledgerPair = await crypto.subtle.generateKey({ name: "ECDSA", namedCurve: "P-256" }, false, ["sign", "verify"]) as CryptoKeyPair;
const ledgerHandle = await signer.es256KeyHandle(ledgerPair, { keyId: "ledger-key" });
assert.ok(ledgerHandle.ok);
const ledgerAnchor = await signer.signV3Anchor({ anchorId: "ledger-anchor", chainId: "audit:ledger", sequence: 0, chainHash: new Uint8Array(32) }, ledgerHandle.value, { anchoredAt: 1_500 });
assert.ok(ledgerAnchor.ok);
console.log(JSON.stringify({ handle: ledgerHandle.ok, anchor: ledgerAnchor.ok, extractable: ledgerPair.privateKey.extractable }));


{"handle":true,"anchor":true,"extractable":false}


## Refuse an extractable private key

The factory enforces non-extractable custody rather than accepting any CryptoKeyPair.


In [ ]:
const extractablePair = await crypto.subtle.generateKey({ name: "Ed25519" }, true, ["sign", "verify"]) as CryptoKeyPair;
const extractableRefusal = await signer.ed25519KeyHandle(extractablePair);
assert.equal(extractableRefusal.ok, false);
if (extractableRefusal.ok) throw new Error("extractable key accepted");
assert.equal(extractableRefusal.error, "invalid_key_handle");
console.log(JSON.stringify({ ok: extractableRefusal.ok, code: extractableRefusal.error }));


{"ok":false,"code":"invalid_key_handle"}


## Define DER signature conversion

A custodian returning DER must convert to the raw r || s encoding required by the key-handle contract. This code uses real local cryptography; it does not emulate a KMS.


In [ ]:
/** ASN.1 DER `SEQUENCE { INTEGER r, INTEGER s }` to the 64-byte raw `r || s` of a P-256 signature. */
function derToRawP256(der: Uint8Array): Uint8Array {
  const malformed = () => new Error("not a DER-encoded P-256 ECDSA signature");
  if (der[0] !== 0x30 || der[1] !== der.length - 2) throw malformed();
  const signatureBytes = new Uint8Array(64);
  let at = 2;
  for (const offset of [0, 32]) {
    const length = der[at + 1] ?? 0;
    if (der[at] !== 0x02 || length < 1 || length > 33 || at + 2 + length > der.length) throw malformed();
    let value = der.subarray(at + 2, at + 2 + length);
    if ((value[0]! & 0x80) !== 0 || (value.length > 1 && value[0] === 0 && (value[1]! & 0x80) === 0)) throw malformed();
    if (value.length === 33) {
      if (value[0] !== 0x00) throw malformed();
      value = value.subarray(1); // the sign byte DER adds when the top bit is set
    }
    signatureBytes.set(value, offset + 32 - value.length); // DER drops leading zero bytes; restore them
    at += 2 + length;
  }
  if (at !== der.length) throw malformed();
  return signatureBytes;
}

/** An ES256 handle over a custodian that signs with ECDSA P-256 SHA-256 and returns DER. */
async function kmssigner.Es256KeyHandle(
  signDer: (message: Uint8Array) => Promise<Uint8Array>, // your KMS call
  spki: Uint8Array,                                      // the key's DER SubjectPublicKeyInfo
  keyId: string,
  role: signer.KeyRole,
): Promise<signer.Es256KeyHandle> {
  const key = await crypto.subtle.importKey("spki", new Uint8Array(spki), { name: "ECDSA", namedCurve: "P-256" }, true, ["verify"]);
  const publicKey = new Uint8Array(await crypto.subtle.exportKey("raw", key)); // 0x04 || x || y
  const b64 = (bytes: Uint8Array) => verifier.utf8Str(verifier.base64urlEncode(bytes));
  const jkt = verifier.v3.thumbprint({ crv: "P-256", kty: "EC", x: b64(publicKey.subarray(1, 33)), y: b64(publicKey.subarray(33)) });
  return {
    sign: async (message) => derToRawP256(await signDer(message)),
    publicKey: () => publicKey,
    thumbprint: () => jkt,
    keyIdentity: () => ({ keyId, publicKey }),
    signingIdentity: () => ({ role, keyId, publicKey }),
  };
}


const convertedWidth = derToRawP256(new Uint8Array(sign("sha256", new TextEncoder().encode("encoding check"), generateKeyPairSync("ec", { namedCurve: "P-256" }).privateKey))).length;
assert.equal(convertedWidth, 64);
console.log(JSON.stringify({ signatureBytes: convertedWidth }));


{"signatureBytes":64}


## Bind a real DER custodian

Import only public SPKI into the handle. The private key remains in the custodian closure.


In [ ]:
const custodian = generateKeyPairSync("ec", { namedCurve: "P-256" });
const signDer = async (signingInput: Uint8Array) => new Uint8Array(sign("sha256", signingInput, custodian.privateKey));
const publicSpki = new Uint8Array(custodian.publicKey.export({ type: "spki", format: "der" }));
const custodyIssuer = await derCustodyHandle(signDer, publicSpki, "custody-issuer", "issuer");
const custodyHolder = createEs256Handle("custody-holder");
assert.equal((await custodyIssuer.publicKey()).length, 65);
console.log(JSON.stringify({ publicKeyBytes: (await custodyIssuer.publicKey()).length }));


{"publicKeyBytes":65}


## Verify a request signed under custody

The independent verifier checks both the converted issuer signature and the holder proof.


In [ ]:
const custodyGrant = await signer.signV3Grant({ issuer: "https://backup.example", grantId: "snapshot-grant",
  audiences: ["https://snapshots.example"], issuedAt: 1_000, notBefore: 1_000, expiresAt: 2_000,
  holderThumbprint: await custodyHolder.thumbprint(), operations: [{ name: "create_snapshot", selectors: [{ kind: "all" }] }],
}, custodyIssuer);
assert.ok(custodyGrant.ok);
const custodyRequest = { method: "POST", targetUri: "https://snapshots.example/snapshots",
  invocationId: "550e8400-e29b-41d4-a716-446655440000", operation: "create_snapshot", castArguments: { t: "null" } as const };
const custodyEnvelope = await signer.signV3Report({ ...custodyRequest, grantCompact: custodyGrant.value.grant }, custodyHolder, { issuedAt: 1_100, proofId: "snapshot-proof" });
assert.ok(custodyEnvelope.ok);
const custodyFacts = verifier.v3.checkEnvelope(custodyEnvelope.value.grant, custodyEnvelope.value.proof, { ...custodyRequest,
  trustedIssuer: { keyId: "custody-issuer", publicKey: await custodyIssuer.publicKey() }, issuer: "https://backup.example", audience: "https://snapshots.example",
  evaluationTime: 1_120, clockSkew: 0, proofMaxAge: 300, nonce: { kind: "not_required" },
});
assert.ok(custodyFacts.ok);
console.log(JSON.stringify({ ok: custodyFacts.ok, operation: custodyFacts.value.operation }));


{"ok":true,"operation":"create_snapshot"}


## Refuse unconverted DER

Returning DER bytes directly violates the raw signature contract.


In [ ]:
const wrongEncoding = { ...custodyIssuer, sign: signDer };
const encodingRefusal = await signer.signV3Anchor({ anchorId: "encoding-anchor", chainId: "audit:encoding", sequence: 0, chainHash: new Uint8Array(32) }, wrongEncoding, { anchoredAt: 1_500 });
assert.equal(encodingRefusal.ok, false);
if (encodingRefusal.ok) throw new Error("DER accepted as raw signature");
assert.equal(encodingRefusal.error, "signing_failed");
console.log(JSON.stringify({ ok: encodingRefusal.ok, code: encodingRefusal.error }));


{"ok":false,"code":"signing_failed"}


## Finish

Shut down the kernel to discard the demonstration keys. No private material was persisted.
Continue with the [getting started guide](../guides/getting-started.md).
